# Generator trip analysis

This notebook analyzes this run's response to the generator trip of `gen14_bus1431_idN` (configured
zero-based generator index `13`): the tripped generator's active power, center-of-inertia (COI) frequencies
for the 3 areas and all 14 zones, and representative surviving 20 kV generators in each. It also writes
derived tables and summaries.

The RAW and XLSX model files are read from the ParaEMT copy in this run's `build/` directory. Generated files
are written to `notebook_outputs/` in the run directory.

In [ ]:
from __future__ import annotations

import csv
import json
import posixpath
import re
import sys
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.feather as pf
from IPython.display import Markdown, display

# DATA_DIR will be set automatically when copied to a run
DATA_DIR = r""

RUN_DIR = Path(DATA_DIR) if DATA_DIR else Path.cwd()
OUTPUT_DIR = RUN_DIR / 'notebook_outputs'
ZONE_OUTPUT_DIR = OUTPUT_DIR / 'zones14'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZONE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ParaEMT's default start time when simulation.oedi_start_timestamp is unset.
DEFAULT_START_TIMESTAMP = '2026-01-01 00:00:00'

# Figures produced by the analysis, keyed by output name, for display below.
FIGURES = {}

print(f'Run:        {RUN_DIR}')
print(f'Output:     {OUTPUT_DIR}')
print(f'Python:     {sys.executable}')

## Recorded streams

The analysis uses two ParaEMT streams. The run's `wiring.json` links say which Recorder holds each one, so
Recorder names do not matter. Timestamps are converted to seconds from the start of the simulation.

In [ ]:
required_streams = {
    5: ('generator_electrical_power', 'generator electrical power'),
    8: ('generator_rotor_speed', 'generator rotor speed'),
}

wiring = json.loads((RUN_DIR / 'wiring.json').read_text(encoding='utf-8'))
components = {component['name']: component for component in wiring['components']}
paraemt = [
    component for component in wiring['components']
    if 'simulation' in component.get('parameters', {})
    and 'powerflow' in component.get('parameters', {})
]
if len(paraemt) != 1:
    raise ValueError(f'Expected one ParaEMT component in wiring.json; found {len(paraemt)}.')
paraemt = paraemt[0]
PARAEMT_BUILD_DIR = RUN_DIR / 'build' / paraemt['name']
MODEL_DIR = PARAEMT_BUILD_DIR / 'models' / '240bus_psse'

recorders = {}
for link in wiring['links']:
    target = components[link['target']]
    if link['source'] == paraemt['name'] and target['type'] == 'Recorder':
        recorders[link['source_port']] = target

start = pd.Timestamp(paraemt['parameters']['simulation'].get('oedi_start_timestamp', DEFAULT_START_TIMESTAMP))
STREAM_FRAMES, stream_rows = {}, []
for number, (stream, description) in required_streams.items():
    if stream not in recorders:
        raise KeyError(f'No Recorder is attached to ParaEMT stream {stream!r}. Add one in the frontend and run again.')
    recorder = recorders[stream]
    path = RUN_DIR / 'build' / recorder['name'] / recorder['parameters']['feather_filename']
    if not path.is_file():
        raise FileNotFoundError(f'{stream}: {path} does not exist.')
    frame = pf.read_feather(path)
    frame['time'] = (pd.to_datetime(frame['time']) - start).dt.total_seconds()
    STREAM_FRAMES[number] = frame
    stream_rows.append({
        'stream': stream,
        'description': description,
        'file': str(path.relative_to(RUN_DIR)),
        'rows': len(frame),
        'time_s': f"{frame['time'].iloc[0]:.3f}-{frame['time'].iloc[-1]:.3f}",
        'size_MB': path.stat().st_size / 1_000_000,
    })

config = json.loads((PARAEMT_BUILD_DIR / 'static_inputs.json').read_text(encoding='utf-8'))
trip = config['gen_trip']
TARGET_CHANNEL = 'gen14_bus1431_idN'
TARGET_GEN_INDEX = 13

assert trip['enabled'], 'Generator trip must be enabled.'
assert int(trip['target_gen']) == TARGET_GEN_INDEX
assert not config['forced_oscillation']['enabled']

display(Markdown('## Generator-trip configuration'))
display(pd.Series(trip, name='value').to_frame())
display(Markdown('## Required recorded streams'))
display(pd.DataFrame(stream_rows).set_index('stream'))

## Analysis code

The scenario configuration is read from the `static_inputs.json` that ParaEMT ran with, in this run's build
directory.

In [ ]:
from pathlib import Path
import argparse
import csv
import json
import re
import zipfile
import xml.etree.ElementTree as ET
import posixpath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Set by configure().
CODE_DIR = HERE = DATA = ROOT = AREA_RAW = MACHINE_RAW = DYNAMIC_XLSX = None


def configure(data_dir, support_dir=None, output_dir=None):
    """Configure scenario inputs, shared model files, and generated outputs."""
    global HERE, DATA, ROOT, AREA_RAW, MACHINE_RAW, DYNAMIC_XLSX
    DATA = Path(data_dir).resolve()
    ROOT = Path(support_dir).resolve() if support_dir is not None else DATA.parent.parent
    HERE = Path(output_dir).resolve() if output_dir is not None else DATA / 'analysis_plots'
    HERE.mkdir(parents=True, exist_ok=True)
    AREA_RAW = ROOT / 'WECC240_2600_2619IBR.raw'
    MACHINE_RAW = ROOT / '240busWECC_2018_PSS_PQLoad_fixedshunt_rm0load_2600_2619G_mod.raw'
    DYNAMIC_XLSX = MACHINE_RAW.with_suffix('.xlsx')


def static_inputs_path():
    paths = sorted(DATA.glob('static_inputs*.json'))
    if len(paths) != 1:
        raise ValueError(f'Expected one static_inputs*.json in {DATA}: {paths}')
    return paths[0]


def read_dynamic_generators(path):
    """Read the column-oriented gen sheet with stdlib OOXML, retaining cell provenance."""
    ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with zipfile.ZipFile(path) as archive:
        strings = []
        if 'xl/sharedStrings.xml' in archive.namelist():
            strings = [''.join(node.itertext()) for node in
                       ET.fromstring(archive.read('xl/sharedStrings.xml')).findall('m:si', ns)]
        workbook = ET.fromstring(archive.read('xl/workbook.xml'))
        sheets = workbook.findall('m:sheets/m:sheet', ns)
        sheet = next(s for s in sheets if s.attrib['name'] == 'gen')
        rid = sheet.attrib['{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id']
        relationships = ET.fromstring(archive.read('xl/_rels/workbook.xml.rels'))
        target = next(r.attrib['Target'] for r in relationships if r.attrib['Id'] == rid)
        target = target.lstrip('/') if target.startswith('/') else posixpath.normpath('xl/' + target)
        cells, labels = {}, {}
        for row in ET.fromstring(archive.read(target)).findall('m:sheetData/m:row', ns):
            for cell in row.findall('m:c', ns):
                v = cell.find('m:v', ns)
                value = v.text if v is not None else ''
                if cell.attrib.get('t') == 's':
                    value = strings[int(value)]
                elif cell.attrib.get('t') == 'inlineStr':
                    value = ''.join(cell.find('m:is', ns).itertext())
                cells[cell.attrib['r']] = value
            label = cells.get('A' + row.attrib['r'], '').strip()
            labels[label] = row.attrib['r']
        records = {}
        for address, value in cells.items():
            match = re.fullmatch(r'([A-Z]+)' + labels['Bus #'], address)
            if not match or match[1] == 'A' or not value:
                continue
            col = match[1]
            key = (int(float(value)), cells[col + labels['Gen id']].strip())
            h_cell = col + labels['H']
            h = float(cells[h_cell])
            model = cells[col + labels['type']].strip()
            if key in records or not np.isfinite(h) or h <= 0 or model != 'GENROU':
                raise ValueError(f'Invalid or duplicate dynamic generator {key}')
            records[key] = dict(H_s=h, dynamic_model=model,
                                inertia_source=path.name, inertia_cell='gen!' + h_cell)
    return records


def read_raw(path):
    """Read bus, generator and area records from this PSS/E RAW file."""
    buses, machines, areas = {}, {}, {}
    section = 'bus'
    for line in path.read_text().splitlines()[3:]:
        if line.strip().startswith('0 /'):
            section = line.split('Begin ', 1)[-1].lower() if 'Begin ' in line else ''
            continue
        if not line.strip() or line.strip() == 'Q':
            continue
        if section != 'bus' and not section.startswith(('generator data', 'area interchange data')):
            continue
        v = next(csv.reader([line], quotechar="'", skipinitialspace=True))
        if section == 'bus':
            buses[int(v[0])] = dict(bus=int(v[0]), bus_name=v[1].strip(),
                                   kv=float(v[2]), area=int(v[4]), zone=int(v[5]))
        elif section.startswith('generator data'):
            machines[int(v[0]), v[1].strip()] = dict(raw_pg_mw=float(v[2]),
                                                    mbase_mva=float(v[8]), status=int(v[14]))
        else:
            areas[int(v[0])] = v[4].strip()
    return buses, machines, areas


def load_csv(number):
    if number not in STREAM_FRAMES:
        raise KeyError(f'Stream {number} was not loaded from this run.')
    df = STREAM_FRAMES[number].copy()
    if not np.isfinite(df.to_numpy(dtype=float)).all() or not (np.diff(df.time) > 0).all():
        raise ValueError(f'Nonfinite data or nonmonotonic time in stream {number}')
    return df


def save(fig, name):
    fig.savefig(HERE / f'{name}.png', dpi=220, bbox_inches='tight')
    fig.savefig(HERE / f'{name}.pdf', bbox_inches='tight')
    FIGURES[name] = fig
    plt.close(fig)


def plot_gen_trip(data_dir=None, support_dir=None, output_dir=None, inertia_csv=None):
    if data_dir is not None:
        configure(data_dir, support_dir, output_dir)
    elif DATA == CODE_DIR:
        raise ValueError('data_dir is required before running the generator-trip analysis')
    buses, _, areas = read_raw(AREA_RAW)
    _, machines, _ = read_raw(MACHINE_RAW)
    dynamics = read_dynamic_generators(DYNAMIC_XLSX)
    power, speed = load_csv(5), load_csv(8)
    if list(power.columns) != list(speed.columns) or not np.array_equal(power.time, speed.time):
        raise ValueError('Power and speed CSVs must have identical channels and timestamps')
    config = json.loads(static_inputs_path().read_text(encoding='utf-8'))
    trip_time = float(config['gen_trip']['time'])
    t = speed.time.to_numpy()
    pre = (t >= trip_time - 0.2) & (t <= trip_time)
    post = t > trip_time
    if not pre.any() or not post.any():
        raise ValueError('Missing pre/post-trip samples')
    # Select by actual sustained loss, not by confusing 0-based and 1-based labels.
    candidates = [c for c in power if '_bus1431_' in c and
                  power.loc[pre, c].mean() > 1000 and np.max(np.abs(power.loc[post, c])) < 1e-5]
    if len(candidates) != 1:
        raise ValueError(f'Cannot uniquely identify tripped machine: {candidates}')
    tripped = candidates[0]
    # The tripped machine leaves the COI from its first sample of sustained zero output, which does
    # not depend on how the stream's timestamps line up with the configured trip time.
    zero_output = (np.abs(power[tripped]) < 1e-5).to_numpy()
    offline = np.zeros(len(zero_output), dtype=bool)
    offline[np.flatnonzero(~zero_output)[-1] + 1:] = True
    records = []
    for column in speed.columns.drop('time'):
        match = re.fullmatch(r'gen(\d+)_bus(\d+)_id(.+)', column)
        if not match:
            raise ValueError(f'Unknown generator channel: {column}')
        number, bus, gid = int(match[1]), int(match[2]), match[3]
        machine = machines[bus, gid]
        if machine['status'] != 1 or machine['mbase_mva'] <= 0:
            raise ValueError(f'Invalid machine metadata: {column}')
        records.append(dict(column=column, generator_number=number, generator_id=gid,
                            **buses[bus], area_name=areas[buses[bus]['area']], **machine,
                            **dynamics[bus, gid],
                            pretrip_p_mw=float(power.loc[pre, column].mean()),
                            mbase_source=MACHINE_RAW.name))
    meta = pd.DataFrame(records).set_index('column', drop=False)
    meta.index.name = 'channel_index'
    if set(dynamics) != set(zip(meta.bus, meta.generator_id)):
        raise ValueError('Dynamic GENROU list and recorded rotor channels do not match exactly')
    h = meta.H_s.copy()
    exact = True
    if inertia_csv is not None:
        inertia_csv = Path(inertia_csv).resolve()
        supplied = pd.read_csv(inertia_csv)
        if supplied.column.duplicated().any():
            raise ValueError('Duplicate inertia channel')
        h = supplied.set_index('column').H_s.reindex(meta.index)
        if not np.isfinite(h).all() or (h < 0).any():
            raise ValueError('Supply finite nonnegative H_s for every recorded machine')
        meta['inertia_source'] = str(inertia_csv)
        meta['inertia_cell'] = 'CSV override'
    meta['H_s'] = h if exact else np.nan
    meta['weight_HS' if exact else 'weight_MBASE_equal_H'] = h * meta.mbase_mva
    meta[['column', 'bus', 'generator_id', 'dynamic_model', 'H_s', 'inertia_source', 'inertia_cell']].to_csv(
        HERE / 'inertia_from_xlsx.csv', index=False)
    # CSV exports absolute electrical rotor angular speed, about 2*pi*60 rad/s.
    frequency = speed.drop(columns='time') / (2 * np.pi)
    if not np.allclose(frequency.loc[pre].median(), 60, atol=0.1):
        raise ValueError('Rotor speed is not consistent with absolute electrical rad/s at 60 Hz')
    weights = h * meta.mbase_mva
    coi = pd.DataFrame({'time_s': t})
    selected, summaries = [], []
    for area, name in areas.items():
        group = meta[(meta.area == area) & (weights > 0)]
        if group.empty:
            raise ValueError(f'No positive-inertia machines in area {area}')
        cols = group.index.tolist()
        w = np.broadcast_to(weights.loc[cols].to_numpy(), (len(t), len(cols))).copy()
        if tripped in cols:
            w[offline, cols.index(tripped)] = 0.0
        if (w.sum(axis=1) <= 0).any():
            raise ValueError('Zero area inertia after trip')
        vals = (frequency[cols].to_numpy() * w).sum(axis=1) / w.sum(axis=1)
        coi[f'area{area}_{name}_Hz'] = vals
        eligible = group[(group.kv == 20.0) & (group.column != tripped)]
        if eligible.empty:
            raise ValueError(f'No surviving 20 kV representative in area {area}')
        representative = eligible.sort_values(['pretrip_p_mw', 'column'], ascending=[False, True]).iloc[0]
        selected.append(representative.to_dict())
        j = np.flatnonzero(post)[np.argmin(vals[post])]
        summaries.append(dict(area=area, area_name=name, recorded_machines=len(group),
                              posttrip_machines=len(group) - int(tripped in cols),
                              minimum_observed_hz=float(vals[j]), minimum_time_s=float(t[j]),
                              end_frequency_hz=float(vals[-1]), representative=representative.column))
    meta.to_csv(HERE / 'generator_metadata.csv', index=False)
    pd.DataFrame(selected).to_csv(HERE / 'representative_generators.csv', index=False)
    coi.to_csv(HERE / 'area_coi_frequency.csv', index=False)
    pd.DataFrame({'time_s': t, 'tripped_P_MW': power[tripped]}).to_csv(HERE / 'tripped_generator_power.csv', index=False)
    reps = pd.DataFrame({'time_s': t})
    angles = pd.DataFrame({'time_s': t})
    for item in selected:
        c = item['column']
        reps[c + '_Hz'] = frequency[c]
        # An optional reconstructed synchronous-frame angle; its unknown initial offset is set to zero.
        delta_omega = speed[c].to_numpy() - 2 * np.pi * 60
        angles[c + '_relative_rad'] = np.r_[0, np.cumsum((delta_omega[1:] + delta_omega[:-1]) * np.diff(t) / 2)]
    reps.to_csv(HERE / 'representative_rotor_frequency.csv', index=False)
    angles.to_csv(HERE / 'reconstructed_relative_rotor_angle.csv', index=False)
    pd.DataFrame({'column': meta.column, 'H_s': meta.H_s}).to_csv(HERE / 'inertia_template.csv', index=False)
    pd.DataFrame(summaries).to_csv(HERE / 'area_summary.csv', index=False)
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.spines.top': False,
                         'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .22,
                         'lines.linewidth': 1.8, 'pdf.fonttype': 42})
    colors = ['#147D92', '#D88120', '#7455A4']
    def setup(ax, ylabel):
        ax.axvline(trip_time, color='#BA3945', ls='--', lw=1.2, label='Trip at 1.000 s')
        ax.set(xlabel='Time (s)', ylabel=ylabel, xlim=(0, t[-1]))
    fig, ax = plt.subplots(figsize=(10, 4.8), layout='constrained')
    ax.plot(t, power[tripped] / 1000, color=colors[0], label='Bus 1431 / ID N (gen14)')
    setup(ax, 'Active power (GW)')
    ax.set_title('Tripped generator active power', loc='left', fontweight='bold')
    ax.text(.04, .18, f'Pre-trip mean: {power.loc[pre, tripped].mean()/1000:.3f} GW\nFirst zero sample: {t[np.flatnonzero(post)[0]]:.3f} s', transform=ax.transAxes)
    ax.legend(loc='upper right')
    save(fig, '01_tripped_generator_active_power')
    fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
    for (area, name), color in zip(areas.items(), colors):
        ax.plot(t, coi[f'area{area}_{name}_Hz'], color=color, label=f'Area {area}: {name}')
    setup(ax, 'Frequency (Hz)')
    title = 'Regional COI frequency | H x MBASE weighted' if exact else 'Regional COI approximation | equal H, MBASE weighted'
    ax.set_title(title, loc='left', fontweight='bold', fontsize=12)
    ax.legend(loc='lower left')
    save(fig, '02_area_coi_frequency')
    fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True, sharey=True, layout='constrained')
    for ax, item, color in zip(axes, selected, colors):
        c, area = item['column'], item['area']
        ax.plot(t, frequency[c], color=color, label='Rotor frequency: omega / (2 pi)')
        ax.plot(t, coi[f'area{area}_{areas[area]}_Hz'], color='#6D7480', ls=':', lw=1.2,
                label='Area COI' if exact else 'Area COI approximation')
        setup(ax, 'Frequency (Hz)')
        ax.set_title(f'Area {area}: {areas[area]} | {item["bus_name"]} | Bus {item["bus"]}, ID {item["generator_id"]}, 20 kV', loc='left', fontsize=11)
        ax.legend(loc='lower left', fontsize=9)
    fig.suptitle('Representative generator rotor frequencies', fontweight='bold')
    save(fig, '03_representative_rotor_frequency')
    report = dict(tripped_channel=tripped, configured_zero_based_index=config['gen_trip']['target_gen'],
                  configured_trip_time_s=trip_time, first_posttrip_sample_s=float(t[np.flatnonzero(post)[0]]),
                  pretrip_mean_mw=float(power.loc[pre, tripped].mean()),
                  posttrip_max_abs_mw=float(np.abs(power.loc[post, tripped]).max()),
                  samples=len(t), sample_interval_s=float(np.median(np.diff(t))),
                  coi_method='H_s * MBASE weighted' if exact else 'equal H assumption; MBASE weighted approximation',
                  rotor_frequency_method='absolute electrical rotor speed / (2*pi); no original angle samples available',
                  area_source=AREA_RAW.name, machine_source=MACHINE_RAW.name, areas=summaries)
    report['inertia_source'] = str(inertia_csv) if inertia_csv else DYNAMIC_XLSX.name + ': gen sheet, H row'
    report['dynamic_generator_count'] = len(dynamics)
    report['all_dynamic_GENROU_channels_covered'] = True
    report['H_range_s'] = [float(h.min()), float(h.max())]
    (HERE / 'summary.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))

In [ ]:
from pathlib import Path
import argparse
import csv
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = None  # set by plot_zones(output_dir=...)


def read_zones(path):
    zones = {}
    active = False
    for line in path.read_text().splitlines():
        if line.strip().startswith('0 /'):
            active = 'Begin Zone data' in line
            continue
        if active and line.strip():
            values = next(csv.reader([line], quotechar="'", skipinitialspace=True))
            zones[int(values[0])] = values[1].strip()
    if set(zones) != set(range(1, 15)):
        raise ValueError(f'Expected exactly 14 zones, found {zones}')
    return zones


def save_zone_figure(fig, name):
    fig.savefig(OUT / (name + '.png'), dpi=200, bbox_inches='tight')
    fig.savefig(OUT / (name + '.pdf'), bbox_inches='tight')
    FIGURES['zones14/' + name] = fig
    plt.close(fig)


def plot_zones(data_dir=None, support_dir=None, output_dir=None):
    global OUT
    if data_dir is not None:
        configure(data_dir, support_dir, Path(output_dir).resolve().parent if output_dir else None)
    elif DATA == CODE_DIR:
        raise ValueError('data_dir is required before running the zone analysis')
    OUT = Path(output_dir).resolve() if output_dir is not None else HERE / 'zones14'
    OUT.mkdir(parents=True, exist_ok=True)
    buses, _, _ = read_raw(AREA_RAW)
    _, machines, _ = read_raw(MACHINE_RAW)
    dynamics = read_dynamic_generators(DYNAMIC_XLSX)
    zones = read_zones(AREA_RAW)
    power, speed = load_csv(5), load_csv(8)
    if list(power.columns) != list(speed.columns) or not np.array_equal(power.time, speed.time):
        raise ValueError('Power/speed channel or time mismatch')
    config = json.loads(static_inputs_path().read_text(encoding='utf-8'))
    trip_time = float(config['gen_trip']['time'])
    time = speed.time.to_numpy()
    pre = (time >= trip_time - .2) & (time <= trip_time)
    post = time > trip_time
    tripped_candidates = [c for c in power if '_bus1431_' in c
                          and power.loc[pre, c].mean() > 1000
                          and np.max(np.abs(power.loc[post, c])) < 1e-5]
    if len(tripped_candidates) != 1:
        raise ValueError('Could not uniquely identify trip')
    tripped = tripped_candidates[0]
    # The tripped machine leaves the COI from its first sample of sustained zero output, which does
    # not depend on how the stream's timestamps line up with the configured trip time.
    zero_output = (np.abs(power[tripped]) < 1e-5).to_numpy()
    offline = np.zeros(len(zero_output), dtype=bool)
    offline[np.flatnonzero(~zero_output)[-1] + 1:] = True
    records = []
    for channel in speed.columns.drop('time'):
        match = re.fullmatch(r'gen(\d+)_bus(\d+)_id(.+)', channel)
        if not match:
            raise ValueError(f'Unknown channel {channel}')
        key = (int(match[2]), match[3])
        records.append(dict(column=channel, generator_id=key[1], **buses[key[0]],
                            **machines[key], **dynamics[key],
                            zone_name=zones[buses[key[0]]['zone']],
                            pretrip_p_mw=float(power.loc[pre, channel].mean()),
                            mbase_source=MACHINE_RAW.name,
                            zone_source=AREA_RAW.name))
    meta = pd.DataFrame(records).set_index('column', drop=False)
    meta.index.name = 'channel_index'
    if set(dynamics) != set(zip(meta.bus, meta.generator_id)):
        raise ValueError('GENROU channels do not cover XLSX generators exactly')
    if not ((meta.H_s > 0) & (meta.mbase_mva > 0) & (meta.status == 1)).all():
        raise ValueError('Invalid inertia, MBASE or status')
    meta['weight_HS'] = meta.H_s * meta.mbase_mva
    frequency = speed.drop(columns='time') / (2*np.pi)
    if not np.allclose(frequency.loc[pre].median(), 60, atol=.1):
        raise ValueError('Unexpected rotor-speed units')
    coi = pd.DataFrame({'time_s': time})
    rep_frequency = pd.DataFrame({'time_s': time})
    selected, summaries = [], []
    for zone, name in zones.items():
        group = meta[meta.zone == zone]
        column = f'zone{zone:02d}_{name}_Hz'
        has_coi = not group.empty
        if has_coi:
            weights = np.broadcast_to(group.weight_HS.to_numpy(), (len(time), len(group))).copy()
            if tripped in group.index:
                weights[offline, group.index.tolist().index(tripped)] = 0
            denominator = weights.sum(axis=1)
            vals = np.divide((frequency[group.index].to_numpy() * weights).sum(axis=1),
                             denominator, out=np.full(len(time), np.nan), where=denominator > 0)
        else:
            vals = np.full(len(time), np.nan)
        coi[column] = vals
        eligible = group[np.isclose(group.kv, 20) & (group.column != tripped)]
        representative = None
        if not eligible.empty:
            representative = eligible.sort_values(['pretrip_p_mw', 'column'], ascending=[False, True]).iloc[0]
            selected.append(representative.to_dict())
            rep_frequency[f'zone{zone:02d}_{representative.column}_Hz'] = frequency[representative.column]
        summaries.append(dict(zone=zone, zone_name=name, synchronous_generators=len(group),
                              surviving_generators=len(group)-int(tripped in group.index),
                              eligible_20kv_generators=len(eligible), coi_available=has_coi,
                              representative_available=representative is not None,
                              representative=representative.column if representative is not None else '',
                              bus=int(representative.bus) if representative is not None else None,
                              generator_id=representative.generator_id if representative is not None else '',
                              bus_name=representative.bus_name if representative is not None else '',
                              kv=float(representative.kv) if representative is not None else None,
                              H_s=float(representative.H_s) if representative is not None else None,
                              end_coi_hz=float(vals[-1]) if np.isfinite(vals[-1]) else None,
                              limitation='' if has_coi and representative is not None else
                              ('No recorded synchronous generators' if not has_coi else 'No surviving 20 kV synchronous generator')))
    meta.to_csv(OUT/'generator_zone_metadata.csv', index=False)
    coi.to_csv(OUT/'zone_coi_frequency.csv', index=False)
    rep_frequency.to_csv(OUT/'representative_rotor_frequency.csv', index=False)
    pd.DataFrame(selected).to_csv(OUT/'representative_generators.csv', index=False)
    pd.DataFrame(summaries).to_csv(OUT/'zone_summary.csv', index=False)
    plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':10, 'axes.spines.top':False,
                         'axes.spines.right':False, 'axes.grid':True, 'grid.alpha':.22,
                         'lines.linewidth':1.7, 'pdf.fonttype':42})
    colors = list(plt.get_cmap('tab20').colors)
    def format_axis(ax, event_linestyle="--"):
        ax.axvline(trip_time, color='#B93B49', lw=1, ls=event_linestyle)
        ax.set(xlim=(0,time[-1]), xlabel='Time (s)', ylabel='Frequency (Hz)')
    fig, ax = plt.subplots(figsize=(13,7), layout='constrained')
    for row in summaries:
        z = row['zone']
        ax.plot(time, coi.iloc[:,z], color=colors[z-1], ls='-',
                label=f'{z:02d} {zones[z]}')
    format_axis(ax, event_linestyle='-')
    ax.set_title('COI frequencies of all 14 zones | H x MBASE weighted', loc='left', fontweight='bold')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01,1), fontsize=9)
    save_zone_figure(fig,'01_all_14_zone_coi_frequency')
    fig_coi, axes_coi = plt.subplots(7,2,figsize=(15,20),sharex=True,sharey=True,layout='constrained')
    fig_rep, axes_rep = plt.subplots(7,2,figsize=(15,20),sharex=True,sharey=True,layout='constrained')
    lookup = {r['zone']:r for r in selected}
    individual = OUT/'individual_zones'
    individual.mkdir(exist_ok=True)
    for row, ax_coi, ax_rep in zip(summaries, axes_coi.flat, axes_rep.flat):
        z = row['zone']
        values = coi.iloc[:,z]
        ax_coi.plot(time, values, color=colors[z-1])
        ax_coi.set_title(f'Zone {z:02d}: {zones[z]} ({row["synchronous_generators"]} synchronous generators)',loc='left',fontsize=10)
        format_axis(ax_coi)
        fig, ax = plt.subplots(figsize=(10,4.8),layout='constrained')
        for target in [ax_rep, ax]:
            target.plot(time, values, color='#656D78', ls='--',lw=1.4,label='Zone COI')
            if z in lookup:
                rep=lookup[z]
                target.plot(time, frequency[rep['column']],color=colors[z-1],
                            label=f'Rotor: bus {rep["bus"]}, ID {rep["generator_id"]} (20 kV)')
                target.set_title(f'Zone {z:02d}: {zones[z]} | {rep["bus_name"]}',loc='left',fontsize=10)
            else:
                target.set_title(f'Zone {z:02d}: {zones[z]}',loc='left')
                target.text(.1,.5,row['limitation'],transform=target.transAxes)
            format_axis(target)
            target.legend(loc='lower left',fontsize=8)
        save_zone_figure(fig, f'individual_zones/zone{z:02d}_coi_and_representative')
    fig_coi.suptitle(f'14-zone COI frequencies | H x MBASE weighted | Trip at {trip_time:g} s',fontweight='bold')
    fig_rep.suptitle(f'20 kV representative rotor frequencies and zone COI | Trip at {trip_time:g} s',fontweight='bold')
    save_zone_figure(fig_coi,'02_zone_coi_panels')
    save_zone_figure(fig_rep,'03_zone_representative_rotor_frequency')
    report=dict(grouping='RAW BUS field 6 (ZONE); all 14 Zone records',
                zone_source=AREA_RAW.name, inertia_source=DYNAMIC_XLSX.name,
                mbase_source=MACHINE_RAW.name, coi_method='H_s * MBASE weighted',
                tripped_channel=tripped, trip_time_s=trip_time,
                selection='Highest mean pre-trip P among surviving recorded 20 kV GENROU in each zone',
                all_105_dynamic_generators_covered=True, zones=summaries,
                unavailable_zones=[r for r in summaries if r['limitation']])
    (OUT/'summary.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    print(pd.DataFrame(summaries).to_string(index=False))
    print('Unavailable zones:', report['unavailable_zones'])

## Run the analysis

In [ ]:
plot_gen_trip(
    data_dir=PARAEMT_BUILD_DIR,
    support_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR,
)
plot_zones(
    data_dir=PARAEMT_BUILD_DIR,
    support_dir=MODEL_DIR,
    output_dir=ZONE_OUTPUT_DIR,
)
print('Generator-trip plot generation complete.')

In [ ]:
power = load_csv(5)
trip_time = float(trip['time'])
pre = (power['time'] >= trip_time - 0.2) & (power['time'] <= trip_time)
post = power['time'] > trip_time

assert TARGET_CHANNEL in power.columns
assert power.columns.get_loc(TARGET_CHANNEL) == TARGET_GEN_INDEX
assert float(power.loc[pre, TARGET_CHANNEL].mean()) > 1000
assert float(power.loc[post, TARGET_CHANNEL].abs().max()) < 1e-5

area_summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
zone_summary = json.loads((ZONE_OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
assert area_summary['tripped_channel'] == TARGET_CHANNEL
assert zone_summary['tripped_channel'] == TARGET_CHANNEL

required_plots = [
    OUTPUT_DIR / '01_tripped_generator_active_power.png',
    OUTPUT_DIR / '02_area_coi_frequency.png',
    OUTPUT_DIR / '03_representative_rotor_frequency.png',
    ZONE_OUTPUT_DIR / '01_all_14_zone_coi_frequency.png',
    ZONE_OUTPUT_DIR / '02_zone_coi_panels.png',
    ZONE_OUTPUT_DIR / '03_zone_representative_rotor_frequency.png',
]
assert all(path.stat().st_size > 10_000 for path in required_plots)
assert len(list((ZONE_OUTPUT_DIR / 'individual_zones').glob('*.png'))) == 14

validation = {
    'status': 'PASS',
    'target_channel': TARGET_CHANNEL,
    'configured_zero_based_index': TARGET_GEN_INDEX,
    'trip_time_s': trip_time,
    'pretrip_mean_MW': float(power.loc[pre, TARGET_CHANNEL].mean()),
    'posttrip_max_abs_MW': float(power.loc[post, TARGET_CHANNEL].abs().max()),
    'individual_zone_plots': 14,
}
display(Markdown('## Validation result'))
display(pd.Series(validation, name='value').to_frame())

In [ ]:
display(Markdown('## Generator-trip and area plots'))
for name in [
    '01_tripped_generator_active_power',
    '02_area_coi_frequency',
    '03_representative_rotor_frequency',
]:
    display(Markdown(f'### `{name}.png`'))
    display(FIGURES[name])

In [ ]:
display(Markdown('## Fourteen-zone plots'))
for name in [
    '01_all_14_zone_coi_frequency',
    '02_zone_coi_panels',
    '03_zone_representative_rotor_frequency',
]:
    display(Markdown(f'### `{name}.png`'))
    display(FIGURES[f'zones14/{name}'])

In [ ]:
tripped_zone = int(next(z['zone'] for z in zone_summary['zones']
                        if z['surviving_generators'] < z['synchronous_generators']))
display(Markdown('## Individual zone plots'))
for zone in range(1, 15):
    name = f'individual_zones/zone{zone:02d}_coi_and_representative'
    note = ' (tripped generator\'s zone)' if zone == tripped_zone else ''
    display(Markdown(f'### `{name}.png`{note}'))
    display(FIGURES[f'zones14/{name}'])

In [ ]:
display(Markdown('## Derived summaries'))
display(Markdown('### Area summary'))
display(pd.read_csv(OUTPUT_DIR / 'area_summary.csv'))
display(Markdown('### Zone summary'))
display(pd.read_csv(ZONE_OUTPUT_DIR / 'zone_summary.csv'))

output_inventory = pd.DataFrame([
    {
        'relative_path': str(path.relative_to(OUTPUT_DIR)),
        'type': path.suffix.lower(),
        'size_kB': path.stat().st_size / 1000,
    }
    for path in sorted(OUTPUT_DIR.rglob('*'))
    if path.is_file()
])
display(Markdown(f'## Output inventory ({len(output_inventory)} files)'))
display(output_inventory)